# Market Basket Analysis
**Dashboard page:** Market Basket
**Tabs:** Co-Purchases · Invoices · Association Rules · Frequent Itemsets · By Material · ML Clustering

**Algorithm:** FP-Growth (mlxtend) for association rules; K-Means/DBSCAN for ML clustering.
**Purpose:** Identify spare parts frequently bought together → bundle promotions, stock suggestions.

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
# Market basket parquet is computed on-demand from the dashboard (not pre-generated).
# To run it: open the dashboard -> Market Basket -> set support/confidence/lift -> click Analyse.
# Once run, results appear below. This notebook shows what the analysis looks like.

import os
basket_paths = ["data/outputs/market_basket.parquet",
                "data/processed/market_basket.parquet",
                "data/interim/market_basket.parquet"]
basket = None
for p in basket_paths:
    if os.path.exists(p):
        basket = pd.read_parquet(p)
        print(f"Loaded: {p} | {basket.shape} | cols: {basket.columns.tolist()}")
        break
if basket is None:
    print("Market basket data not yet generated.")
    print("Run the analysis from the dashboard first: Market Basket -> set parameters -> Analyse.")
    print()
    print("Expected output columns (pairs/rules):")
    print("  antecedents, consequents, support, confidence, lift, material_names")


In [ ]:
# Reconstruct basket from orders if dashboard output not yet available
orders = load("orders_clean.parquet")
sp = orders[orders["mc_category"]=="Spare Parts"].copy() if "mc_category" in orders.columns else orders.copy()
doc_col = "Sales Document" if "Sales Document" in sp.columns else None
mat_col = "Material"       if "Material"       in sp.columns else None

if doc_col and mat_col:
    baskets = sp.groupby(doc_col)[mat_col].apply(list)
    from collections import Counter
    pair_counts = Counter()
    for items in baskets:
        uniq = list(set(items))
        for i in range(len(uniq)):
            for j in range(i+1,len(uniq)):
                pair = tuple(sorted([uniq[i],uniq[j]]))
                pair_counts[pair]+=1
    top_pairs = pd.DataFrame([(a,b,cnt) for (a,b),cnt in pair_counts.most_common(30)],
                              columns=["item_a","item_b","count"])
    print(f"Total invoices analysed : {len(baskets):,}")
    print(f"Unique item pairs found : {len(pair_counts):,}")
    print()
    print("Top 30 co-purchased pairs:")
    print(top_pairs.to_string(index=False))

    fig,ax = plt.subplots(figsize=(12,6))
    top_pairs.head(15).set_index("item_a")["count"].sort_values().plot(
        kind="barh",ax=ax,color=PALETTE[0],edgecolor="white")
    ax.set_title("Top 15 Co-Purchased Spare Part Pairs
(by co-occurrence count across invoices)")
    ax.set_xlabel("Co-purchase count"); plt.tight_layout(); plt.show()
else:
    print("Could not find Sales Document / Material columns in orders_clean.")


In [ ]:
if basket is not None:
    # Display actual market basket results
    print(basket.head(20).to_string())
    if "lift" in basket.columns:
        fig,axes = plt.subplots(1,2,figsize=(13,5))
        basket["lift"].hist(bins=40,ax=axes[0],color=PALETTE[0],edgecolor="white",alpha=0.8)
        axes[0].axvline(1,color=PALETTE[1],ls="--",lw=1.5,label="Lift=1 (no association)")
        axes[0].set_title("Association Rule Lift Distribution
(lift>1 = items bought together more than by chance)")
        axes[0].set_xlabel("Lift"); axes[0].legend()
        if "confidence" in basket.columns and "support" in basket.columns:
            axes[1].scatter(basket["support"],basket["confidence"],c=basket["lift"],
                            cmap="viridis",alpha=0.6,s=25)
            axes[1].set_title("Support vs Confidence (coloured by Lift)")
            axes[1].set_xlabel("Support"); axes[1].set_ylabel("Confidence")
        plt.tight_layout(); plt.show()


**Parameters used in the dashboard:**
- `min_support` (default 0.01): minimum fraction of invoices containing the itemset
- `min_confidence` (default 0.20): P(B|A) — how often B appears when A is bought
- `min_lift` (default 1.5): how much more likely than random co-occurrence